# W10 · Day 2 — KB Lifecycle: Tombstones, the IsNull Bug, KPI Wrap

**~90 minutes · in-class demo · Jupyter notebook · Track A**

**⚠ Still DANGER ZONE.** Yesterday: caching (Layer 1 semantic + Layer 2
provider). Today: the lifecycle side — what happens when documents change,
how tombstones preserve history, and a silent Qdrant filter bug we'll
diagnose live before fixing.

**The teaching arc:**
- Naive re-ingestion creates duplicates (Cell 2)
- Tombstones are the fix concept (Cell 3)
- **We'll write the tombstone filter WRONG first** using `IsNullCondition`
  — it will silently return 0 matches (Cell 4)
- Diagnose live, understand Qdrant's field-absent vs field-null semantics
  (Cell 5), fix with `IsEmptyCondition` (Cell 6)
- End-to-end update flow (Cell 7)
- **Cache invalidation gotcha** — the semantic cache from Day 1 now serves
  stale answers (Cell 8)
- Re-indexing + versioning + drift + cold start — brief conceptual (Cells 9-11)
- KPI comparison W8→W9→W10 (Cell 12)

**Cost per full run:** ~$0.01

**Prerequisite:** Day 1 ran cleanly. We reuse the same Acme corpus.

---

## Cell 1 — Setup + fresh Qdrant collection

Fresh collection each session for reproducibility. In your capstone you
wouldn't recreate the collection — you'd operate on `capstone_chunks_v2`
in place. For the demo notebook, fresh state = clean teaching.

In [ ]:
import os
import sys
import time
from pathlib import Path

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY"
assert os.environ.get("QDRANT_URL"),     "Set QDRANT_URL"
assert os.environ.get("QDRANT_API_KEY"), "Set QDRANT_API_KEY"

sys.path.insert(0, str(Path.cwd()))

from wk10_pipeline import (
    load_acme_corpus, load_test_queries,
    make_live_filter, make_broken_filter,
    cache_init, cache_clear, cache_put, cache_lookup,
    build_rag_prompt, count_tokens, estimate_cost_usd,
)

from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    FieldCondition, Filter, MatchValue,
    IsNullCondition, IsEmptyCondition, PayloadField,
    PayloadSchemaType,
)

corpus  = load_acme_corpus()
queries = load_test_queries()
openai_client = OpenAI()
qdrant = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])
COLLECTION = "wk10_day2_acme"

# Recreate collection
try:    qdrant.delete_collection(COLLECTION)
except: pass
qdrant.create_collection(COLLECTION,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE))

# CRITICAL: Qdrant requires a payload index on any field you want to
# filter on. This mirrors ensure_payload_indexes() in the shipped
# src/rag/pipeline.py. Without these, Cells 4-7 filter operations
# will return HTTP 400 'Index required but not found'.
for field, schema in [
    ("source",     PayloadSchemaType.KEYWORD),
    ("deleted_at", PayloadSchemaType.FLOAT),
]:
    try:
        qdrant.create_payload_index(
            collection_name=COLLECTION,
            field_name=field,
            field_schema=schema,
            wait=True,
        )
    except Exception:
        pass  # already exists — safe
print(f"✓ Payload indexes ensured: source (keyword), deleted_at (float)")

def embed_texts(texts):
    return [item.embedding for item in openai_client.embeddings.create(
        model="text-embedding-3-small", input=texts).data]

# Ingest — NO deleted_at field set (this is critical for Cell 4's bug)
vectors = embed_texts([d["text"] for d in corpus])
points = [
    PointStruct(
        id=i,
        vector=vec,
        payload={**doc, "source": doc["id"], "ingested_at": time.time()},
        # NOTE: deliberately NO deleted_at field. Matches what naive
        # ingestion does — the field only appears when tombstoned.
    )
    for i, (doc, vec) in enumerate(zip(corpus, vectors))
]
qdrant.upsert(COLLECTION, points=points, wait=True)
print(f"✓ Ingested {len(points)} chunks into {COLLECTION!r} (no deleted_at set)")

# Verify
total = qdrant.count(COLLECTION, exact=True).count
print(f"✓ Total points in collection: {total}")

---

## Cell 2 — The document update problem

**Scenario:** the `feat_dashboards` doc gets updated. Product team added
PDF export as a new sub-feature. What happens if we just re-ingest naively?

Watch retrieval get confused by duplicate versions.

In [ ]:
# Simulate the updated feat_dashboards doc — new paragraph about PDF export
updated_dashboards_doc = {
    "id": "feat_dashboards",
    "title": "Dashboards",
    "category": "feature",
    "text": (
        "Build interactive visualisations from your data warehouse without writing "
        "SQL. Drag-and-drop chart builder supports bar, line, area, scatter, and "
        "heatmap types. Dashboards can be embedded into external applications via "
        "signed iframes. Row-level security policies control which slices of data "
        "each user can see. "
        # NEW paragraph — the update:
        "Any dashboard can be exported to PDF via the toolbar button — layout is "
        "preserved and each chart renders at its display resolution. Exports "
        "respect row-level security so recipients see only their permitted data."
    ),
}

# NAIVE re-ingest — just upsert with a new id. No cleanup of old chunk.
new_vec = embed_texts([updated_dashboards_doc["text"]])[0]
qdrant.upsert(COLLECTION, points=[
    PointStruct(
        id=100,  # different id from the original (id=0)
        vector=new_vec,
        payload={**updated_dashboards_doc, "source": "feat_dashboards",
                 "ingested_at": time.time()},
    )
], wait=True)

print(f"✓ Naive re-ingest done. Collection count: {qdrant.count(COLLECTION, exact=True).count}")
print(f"  (Was 15, now 16 — we added the new version WITHOUT removing the old.)\n")

# Now query for the new feature. Retrieval should return BOTH versions.
query = "how do I export a dashboard to PDF"
q_vec = embed_texts([query])[0]
hits = qdrant.query_points(COLLECTION, query=q_vec, limit=5).points

print(f"Query: {query!r}\n")
print(f"  {'Rank':<5s} {'Point ID':<10s} {'Source':<20s} {'Preview'}")
for i, h in enumerate(hits, 1):
    src = h.payload.get("source", "?")
    text_preview = h.payload.get("text", "")[:50]
    dup_marker = " ← DUPLICATE" if src == "feat_dashboards" else ""
    print(f"  {i:<5d} {str(h.id):<10s} {src:<20s} {text_preview}...{dup_marker}")

**The problem is now visible:** two versions of the `feat_dashboards` doc
in the top hits. Retrieval doesn't know one is stale. The LLM will see
both, waste tokens, and might synthesise across the two versions in
inconsistent ways.

**Naive fixes that don't work:**
- Reuse the same point id — you'd overwrite, but then eval reruns pointing
  at old chunk_ids get the NEW content silently (audit trail broken)
- Delete old points before inserting new — you lose forensic history and
  can't reproduce last month's evaluation

**The right fix: tombstones.** Mark old chunks as deleted (with a timestamp)
but keep them around. Filter them out at query time.

---

## Cell 3 — Tombstone concept

**"Every time you delete a chunk, you delete a witness."** (Deck slide 23.)

**The pattern:**
1. Add `deleted_at: <timestamp>` to the payload of the old chunk
2. Leave the point itself in Qdrant (vector still there, chunk_id still valid)
3. Filter it out at query time with a `must=[IsEmpty(deleted_at)]` filter

**Cost:** ~50 bytes per dead chunk (one epoch timestamp). Trivial at
capstone scale (400 chunks × a few updates per year = negligible).

**Benefit:**
- Eval reproducibility — re-run last month's audit against last month's KB state
  by setting `include_deleted=True` and filtering by `deleted_at > last_month`
- Rollback — un-tombstone is one line (`set_payload(deleted_at=None)`);
  un-delete needs full re-ingest
- Forensics — "when did this chunk disappear?" becomes a queryable question

Now we implement it. We'll write it WRONG first.

---

## Cell 4 — Live bug demo: the IsNull trap

**First attempt:** write `tombstone_source()` using `IsNullCondition`. This
is what most learners write first — the Qdrant API has an `IsNullCondition`
and "deleted_at is null" reads intuitively like SQL.

**Predicted behaviour:** it should tombstone all the live chunks from a
given source, then re-run count should show them tombstoned.

**Actual behaviour (spoiler):** 0 chunks tombstoned. Silent failure.
Let's watch it happen.

In [ ]:
# The naive (BROKEN) implementation using IsNullCondition
def tombstone_source_BROKEN(source_name: str) -> int:
    """Mark all live chunks from source as tombstoned. USES IsNullCondition."""
    flt = Filter(must=[
        FieldCondition(key="source", match=MatchValue(value=source_name)),
        IsNullCondition(is_null=PayloadField(key="deleted_at")),
    ])
    # Count matching BEFORE the set_payload — that's how many we'll tombstone
    pre_count = qdrant.count(COLLECTION, count_filter=flt, exact=True).count
    print(f"    Matched by IsNull filter: {pre_count} chunks")
    
    qdrant.set_payload(
        collection_name=COLLECTION,
        payload={"deleted_at": time.time()},
        points=flt,
        wait=True,
    )
    return pre_count

# We want to tombstone the OLD feat_dashboards chunk (point id=0).
# There should be 1 chunk with source=feat_dashboards AND no deleted_at.
print("══ Attempt 1: tombstone using IsNullCondition ══\n")
print("Trying to tombstone the old feat_dashboards chunk (point id=0)...")
n = tombstone_source_BROKEN("feat_dashboards")
print(f"\nReturned: {n} chunks tombstoned")
print(f"\n💥 Silent failure. IsNull matched 0 chunks — nothing got tombstoned.")
print(f"But we KNOW point id=0 exists AND has source=feat_dashboards AND has no deleted_at.")
print(f"So why did the filter miss it?")

**Diagnose it live** — what does `IsNullCondition` actually match?

In [ ]:
# Let's understand what IsNullCondition does by controlled experiment.
# We'll set deleted_at=None (explicit null) on ONE point and see if IsNull
# catches it. This is the key semantic distinction.

print("══ Controlled experiment: what does IsNullCondition actually match? ══\n")

# Point id=1 gets deleted_at=None explicitly set
qdrant.set_payload(
    collection_name=COLLECTION,
    payload={"deleted_at": None},
    points=[1],
    wait=True,
)
print("Set deleted_at=None on point id=1 (explicit null value)")

# Now count with IsNull — expect it to match id=1 only
n_isnull = qdrant.count(
    COLLECTION,
    count_filter=Filter(must=[IsNullCondition(is_null=PayloadField(key="deleted_at"))]),
    exact=True,
).count
print(f"IsNullCondition matches: {n_isnull} point(s)")

# And with IsEmpty — expect it to match all live points (everyone without deleted_at
# set to a real timestamp, PLUS point id=1 which has deleted_at=None)
n_isempty = qdrant.count(
    COLLECTION,
    count_filter=Filter(must=[IsEmptyCondition(is_empty=PayloadField(key="deleted_at"))]),
    exact=True,
).count
print(f"IsEmptyCondition matches: {n_isempty} point(s)")

# Clean up — remove the deleted_at=None from point 1
qdrant.delete_payload(
    collection_name=COLLECTION,
    keys=["deleted_at"],
    points=[1],
    wait=True,
)

**The diagnosis** (this is the pedagogical peak of the week):

| Condition | Matches when field is... |
|---|---|
| `IsNullCondition` | **present with value `null`** (only) |
| `IsEmptyCondition` | **absent OR null OR empty list** |

Fresh ingestion doesn't set `deleted_at` at all — the field is **absent**
from the payload. `IsNull` requires the field to be present with value
`null`, so it returns False for absent fields. Our tombstone_source
matched zero chunks and silently did nothing.

**Two ways to fix:**
1. Change the filter to `IsEmptyCondition` (defensive — works regardless
   of ingestion behaviour). **This is the right fix.**
2. Change the ingestion to always set `deleted_at: None` explicitly at
   insert time (fragile — any future ingestion path that forgets breaks
   retrieval silently). **Don't do this.**

**Which failure mode is worse?** The IsNull bug is silent — no exception,
no warning, retrieval just returns fewer results than expected. Silent
bugs in retrieval erode trust invisibly. The IsEmpty fix is defensive:
it works even if ingestion changes underneath you.

---

## Cell 5 — Note on the audit

This wasn't hypothetical. The W10 deck shipped with `IsNullCondition` in
the code slides (slides 23-24). An external auditor caught it in review.

The **shipped reference code** for W10 sidestepped the bug by setting
`deleted_at: None` explicitly at ingestion — so it worked in practice. But
the pattern was fragile: any learner extending it (a new field like
`archived_at`, a new ingestion script that forgot the `None` assignment)
would silently break retrieval.

**The deck v2 (and this notebook, and the shipped code from W10 v2)
uses `IsEmptyCondition`.** Self-sufficient — no implicit contract with
ingestion required.

**Lesson:** on operational reliability weeks, prefer defensive primitives
even when the offensive one would work today. The maintainer six months
from now — who might be you — will thank you.

---

## Cell 6 — The IsEmpty fix

Rewrite `tombstone_source` using `IsEmptyCondition`. Should just work.

In [ ]:
def tombstone_source(source_name: str) -> int:
    """Mark all live chunks from source as tombstoned. USES IsEmptyCondition.
    
    IsEmpty matches whether deleted_at is absent, null, or empty list —
    all three represent "not yet tombstoned" in practice.
    """
    flt = Filter(must=[
        FieldCondition(key="source", match=MatchValue(value=source_name)),
        IsEmptyCondition(is_empty=PayloadField(key="deleted_at")),
    ])
    pre_count = qdrant.count(COLLECTION, count_filter=flt, exact=True).count
    
    qdrant.set_payload(
        collection_name=COLLECTION,
        payload={"deleted_at": time.time()},
        points=flt,
        wait=True,
    )
    return pre_count

print("══ Attempt 2: tombstone using IsEmptyCondition ══\n")
n = tombstone_source("feat_dashboards")
print(f"✓ Tombstoned {n} chunk(s) from source=feat_dashboards\n")

# Verify — the OLD point (id=0) should now have deleted_at set;
# the NEW point (id=100) should NOT (we inserted it after the tombstone would have run).
# Actually wait — we inserted id=100 in Cell 2 BEFORE running tombstone. So id=100 was
# also live when tombstone ran. Let's check what actually happened:
print("Current state of feat_dashboards chunks:")
scroll_result, _ = qdrant.scroll(
    COLLECTION,
    scroll_filter=Filter(must=[FieldCondition(key="source", match=MatchValue(value="feat_dashboards"))]),
    limit=10,
    with_payload=True,
)
for p in scroll_result:
    deleted = p.payload.get("deleted_at")
    status = f"tombstoned at {deleted:.0f}" if deleted else "LIVE"
    print(f"  id={p.id}  status={status}  text={p.payload['text'][:50]}...")

**Discussion moment:** notice both feat_dashboards chunks got tombstoned
(id=0 old, id=100 new) — because both were LIVE when we ran tombstone_source.

**This reveals the correct update sequence:**
1. Tombstone all live chunks for the source (marks them dead)
2. THEN insert the new chunks (they start live)

**Wrong order** (insert first, tombstone second) tombstones the new chunk
you just inserted. Order matters. Cell 7 shows the correct flow end-to-end.

---

## Cell 6b — Retrieval-time filter with IsEmpty

Now every query should filter tombstoned chunks by default. Wire it in.

In [ ]:
def dense_retrieve_v10(query: str, k: int = 5, include_deleted: bool = False):
    """Retrieval with tombstone filter applied by default.
    
    include_deleted=True is the audit escape hatch — set only when re-running
    historical evaluations that need the KB as it existed then.
    """
    q_vec = embed_texts([query])[0]
    query_filter = None if include_deleted else make_live_filter()
    hits = qdrant.query_points(
        COLLECTION, query=q_vec, limit=k, query_filter=query_filter,
    ).points
    return [{"id": h.id, "score": h.score, "doc": h.payload} for h in hits]

query = "how do I export a dashboard to PDF"

print(f"Query: {query!r}\n")

print("WITHOUT tombstone filter (include_deleted=True):")
for h in dense_retrieve_v10(query, k=3, include_deleted=True):
    dead = h["doc"].get("deleted_at") is not None
    mark = " [TOMBSTONED]" if dead else ""
    print(f"  id={h['id']}  score={h['score']:.4f}  {h['doc'].get('source', '?')}{mark}")

print("\nWITH tombstone filter (default — include_deleted=False):")
for h in dense_retrieve_v10(query, k=3):
    print(f"  id={h['id']}  score={h['score']:.4f}  {h['doc'].get('source', '?')}")

print("\nExpected: the second list excludes both tombstoned dashboards chunks.")
print("Live retrieval returns only currently-live docs.")

---

## Cell 7 — End-to-end update flow

The correct sequence: tombstone old → insert new → validate.

This is the pattern `ingest_or_update_source()` implements in the shipped
capstone code.

In [ ]:
def ingest_or_update_source(source: str, doc: dict) -> dict:
    """Correct update flow:
      1. Tombstone all live chunks for source
      2. Insert the new doc as fresh chunk(s)
      3. Return {tombstoned, ingested} counts
    """
    n_tombstoned = tombstone_source(source)
    
    # Insert fresh — new point ids, no deleted_at
    vec = embed_texts([doc["text"]])[0]
    # Use a fresh id derived from time to avoid collisions with existing ids
    new_id = int(time.time() * 1000) % 100000
    qdrant.upsert(COLLECTION, points=[
        PointStruct(
            id=new_id, vector=vec,
            payload={**doc, "source": source, "ingested_at": time.time()},
        )
    ], wait=True)
    return {"tombstoned": n_tombstoned, "ingested": 1, "new_id": new_id}

# Simulate ANOTHER update — feat_alerts gets a new capability
updated_alerts = {
    "id": "feat_alerts", "title": "Alerting", "category": "feature",
    "text": ("Get notified when metrics cross thresholds. Absolute values, "
             "percentage changes, anomaly detection. Channels: email, Slack, "
             "PagerDuty, webhooks. NEW: multi-condition alerts with AND/OR logic "
             "— combine 'error rate > 5%' AND 'traffic > 1000 rpm' into one alert."),
}

print("══ End-to-end update: feat_alerts gets multi-condition alerts ══\n")
result = ingest_or_update_source("feat_alerts", updated_alerts)
print(f"Update result: {result}\n")

# Verify — live query should return the NEW version
q = "can I combine multiple conditions in one alert"
print(f"Post-update query: {q!r}")
hits = dense_retrieve_v10(q, k=2)
for h in hits:
    print(f"  id={h['id']}  {h['doc'].get('source','?'):<20s} {h['doc']['text'][:60]}...")

---

## Cell 8 — Cache invalidation gotcha

**You just updated a document. But the semantic cache from Day 1 doesn't
know.** It has the OLD answer indexed by question embedding, and any
similar question hits it — serving stale data.

**Watch it happen:**

In [ ]:
# Set up a fresh cache and prime it with an answer about alerts BEFORE the update
cache_db = "wk10_day2_cache.db"
if os.path.exists(cache_db):
    os.remove(cache_db)
cache = cache_init(cache_db)

# Simulate: user asks the question BEFORE the multi-condition feature was added.
# Answer is based on OLD alerts doc.
stale_answer = ("Alerts trigger on a single threshold — absolute value, percentage "
                "change, or anomaly detection. Combining conditions is not supported.")
old_question = "can I set up complex alerts with multiple conditions"
old_q_vec = embed_texts([old_question])[0]
cache_put(cache, old_question, old_q_vec, stale_answer)
print(f"✓ Cache primed with STALE answer about alerts:\n  {stale_answer}\n")

# Now user asks a paraphrase AFTER the update went live
new_question = "can I combine multiple conditions in one alert"
new_q_vec = embed_texts([new_question])[0]
hit = cache_lookup(cache, new_q_vec, threshold=0.85)  # low threshold to force hit

print(f"User asks: {new_question!r}")
print(f"Cache lookup (t=0.85): {'HIT' if hit else 'miss'}")
if hit:
    print(f"  Similarity: {hit['similarity']:.4f}")
    print(f"  Matched question: {hit['matched_question']!r}")
    print(f"  Returned answer: {hit['answer']}\n")
    print(f"💥 Cache served the STALE answer.")
    print(f"The Qdrant KB has the new feature. The cache doesn't. Users get old info.")

In [ ]:
# The fix: clear the cache as part of the update runbook.
# In production this could be more surgical (only invalidate entries whose
# retrieved chunks came from the updated source) but full-clear is simple + safe.

print("══ Cache invalidation as part of the update flow ══\n")
n_cleared = cache_clear(cache)
print(f"✓ Cleared {n_cleared} cache entries\n")

# Now the same question misses the cache → runs full RAG → gets fresh answer
hit = cache_lookup(cache, new_q_vec, threshold=0.85)
print(f"Post-clear cache lookup: {'HIT' if hit else 'miss'}")
print(f"\nOn miss, the caller falls through to full RAG, which retrieves the")
print(f"NEW alerts doc from Qdrant (with the multi-condition feature) and")
print(f"generates a fresh answer. Then that fresh answer gets cached.\n")

print(f"Enshrine this in the update runbook:")
print(f"  1. Tombstone old chunks")
print(f"  2. Insert new chunks")
print(f"  3. Clear semantic cache  ← don't forget")
print(f"  4. Bump kb_version (Cell 10)")

---

## Cell 9 — Re-indexing (verbal + corrected cost formula)

**When does re-indexing happen?** Three triggers:
1. **Embedding model change** — swapping `text-embedding-3-small` → `-large`
   for better retrieval
2. **Chunker strategy change** — different boundaries, different sizes,
   different overlap
3. **Large corpus migration** — new collection, different schema

**Cost formula (deck v2 slide 31, post-audit):**
```
cost = chunks × avg_tokens_per_chunk × price_per_token
```

The original deck expressed cost "per 100K chunks" without stating
tokens/chunk, which was ~20× off for realistic 200-token chunks. Always
state the assumption.

In [ ]:
# Concrete estimate for the capstone
for n_chunks, avg_tokens, model in [
    (400, 200, "text-embedding-3-small"),      # our capstone
    (400, 200, "text-embedding-3-large"),      # upgrade
    (100_000, 200, "text-embedding-3-small"),  # medium enterprise
    (100_000, 500, "text-embedding-3-small"),  # larger chunks
]:
    total_tokens = n_chunks * avg_tokens
    from wk10_pipeline import PRICING
    price = PRICING[model]
    cost = total_tokens * price / 1_000_000
    print(f"  {n_chunks:>6d} chunks × {avg_tokens:>4d} tok × {model}: "
          f"~${cost:.4f}")

print(f"\n(Prices Aug 2026: 3-small=$0.02/M, 3-large=$0.13/M — verify before large runs.)")

---

## Cell 10 — kb_version field + audit-replay concept

**The pattern:** tag every chunk with a `kb_version` at ingestion. Bump
the version whenever you re-index or make a large-scale corpus change.

**Why:** lets you filter retrieval to a specific version for A/B tests
or audit replay:

```python
# Compare retrieval quality between last quarter's KB and current
audit_filter = Filter(must=[FieldCondition(
    key="kb_version", match=MatchValue(value="2026-Q2"))])
hits = qdrant.query_points(..., query_filter=audit_filter, ...)
```

**Naming convention** (programme default): `YYYY-Q<N>` for regular
quarterly refreshes; `YYYY-MM-DD-adhoc` for one-off changes.

We're not running this cell — sketch only. `kb_version` is a payload
field you set at ingestion. Nothing runtime-magical about it.

---

## Cell 11 — Cold start + data drift (verbal)

**Cold start:** you've just deployed. No cache entries. Every query pays
full RAG cost. **First-week costs will be higher than steady-state.**
Mitigation: pre-warm the cache with synthetic questions from FAQ / support
history if you have them. (Stretch lab option.)

**Data drift:** the KB changes (updates happen), and cached answers
gradually become stale even for questions that don't obviously map to
the changed docs. **Mitigation:**
- TTL on cache entries (delete anything older than 30 days)
- Sample-based validation (weekly: pull 20 cache hits, re-run through
  full RAG, compare answers)
- Clear on every document update (what we did in Cell 8 — blunt but safe)

**Embedding drift** (a different concern): if OpenAI updates the embedding
model, existing embeddings are in a different space than new ones. Rare
but has happened. Mitigation: pin your embedding model version explicitly.
This is why W7 fixed us to `text-embedding-3-small` (not `text-embedding-3`).

---

## Cell 12 — KPI comparison across W8 → W9 → W10

Where the whole 3-week arc lands. Column meanings:
- **W8 baseline:** pure dense retrieval, no cache, blind ingestion
- **W9 baseline:** hybrid + cross-encoder rerank, still no cache
- **W10 warm state:** hybrid + rerank + semantic cache (warm)

In [ ]:
# Illustrative numbers — real ones come from the learner's own wk10-snapshot.md.
# These are typical ranges observed on the Acme corpus + 6 test queries.
kpi_table = [
    ("Precision@3",           "0.67",   "0.83",   "0.83",   "W9 lift"),
    ("Recall@3",              "0.67",   "0.83",   "0.83",   "tracks precision here"),
    ("Grounded response rate","0.85",   "0.90",   "0.90",   "cache doesn't affect"),
    ("Cost per query (cold)", "$0.0004", "$0.0004", "$0.0004", "first call same"),
    ("Cost per query (warm)", "$0.0004", "$0.0004", "$0.00003", "90% cache hit"),
    ("Latency p50 (cold)",    "800ms",  "850ms",  "850ms",  "rerank ~50ms"),
    ("Latency p50 (warm)",    "800ms",  "850ms",  "15ms",   "cache eliminates LLM call"),
    ("Eval reproducibility",  "broken", "broken", "solved", "tombstones"),
]

print(f"══ KPI arc across W8 → W9 → W10 (illustrative) ══\n")
print(f"  {'KPI':<26s}  {'W8':>10s}  {'W9':>10s}  {'W10 warm':>10s}  {'notes'}")
print(f"  {'-'*26}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*20}")
for kpi, w8, w9, w10, note in kpi_table:
    print(f"  {kpi:<26s}  {w8:>10s}  {w9:>10s}  {w10:>10s}  {note}")

print(f"\nThe operational + economic story: W9 was the algorithmic win")
print(f"(retrieval quality). W10 is the operational win (cost + latency")
print(f"+ reproducibility). Both matter. Neither replaces the other.")
print(f"\nYour own numbers land in Track B — measure them, don't guess.")

---

## Cell 13 — Wrap + Core/Stretch hand-off + W11 preview

**Today (Day 2) you built:**
1. Saw the naive-update problem (duplicates in retrieval)
2. Diagnosed the IsNull silent bug live — memorable
3. Fixed it with IsEmptyCondition (Qdrant semantics: absent OR null OR empty)
4. End-to-end update flow (tombstone → insert → verify)
5. Cache invalidation gotcha (blunt-clear-on-update = safe default)
6. Re-indexing cost formula (chunks × tokens × price)
7. kb_version + drift + cold start (conceptual)
8. Full-week KPI story

---

## Track B — Core + Stretch (continuing from W9)

Open `AI-RAG_W10_Application_Growth_Guide.md`.

**🟢 CORE (mandatory · ~2.5 hours):**
- Step 1: Wire semantic cache into `qdrant_rag.py`; re-run golden set cold + warm
- Step 2: Add tombstone update flow to `pipeline.py`; pick a capstone doc,
  update it, verify tombstone + fresh insert + cache clear
- Step 3: Update `wk10-snapshot.md` with cost/latency deltas vs W9
- Step 4: Extend ADR with W10 decisions (cache backend, threshold, tombstone semantics)

**🟡 STRETCH (optional · ~45 min):**
- Prompt compression via LLMLingua on 5 golden questions, OR
- Cold-start synthetic Q+A generator: pick 20 questions the corpus should
  answer, embed + cache them before deployment

---

**W11 preview:** RAG Evaluation Framework. We build a proper eval harness
with pytest, regression tests, and CI-friendly golden-set validation. The
cache and tombstone patterns from this week become things the eval harness
explicitly tests ("did the cache invalidation actually happen?").

The DANGER ZONE ends here. W11 and W12 are calmer weeks — evaluation
harness and debugging patterns. Consolidate this weekend.